In [ ]:
import sys
from pathlib import Path
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
PROJECT_ROOT = Path.home() / "si630_nlp_project"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    PROCESSED_DATA_DIR,
    INTERIM_DATA_DIR,
    BASELINE_MODELS_DIR,
    REPORTS_DIR,
    TABLE_DIR,
)

In [ ]:
DATASET_TAG = "large_lite"
EXPERIMENT_TAG = "finbert_v1"

TEXT_COL = "full_text"
LABEL_COL = "label_5d"

MODEL_NAME = "ProsusAI/finbert"

MAX_LENGTH = 512
CHUNK_BODY_LEN = 510
MAX_CHUNKS_PER_DOC = 12
BATCH_SIZE = 8

FORCE_CPU = False
DEVICE = "cpu" if FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

In [ ]:
BASELINE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

model_path = BASELINE_MODELS_DIR / f"finbert_embedding_logistic_{DATASET_TAG}_{EXPERIMENT_TAG}.joblib"
report_path = REPORTS_DIR / f"finbert_embedding_logistic_{DATASET_TAG}_{EXPERIMENT_TAG}_results.json"
val_table_path = TABLE_DIR / f"finbert_embedding_logistic_{DATASET_TAG}_{EXPERIMENT_TAG}_validation_results.csv"

train_embed_path = INTERIM_DATA_DIR / f"train_doc_embedding_finbert_{DATASET_TAG}_{EXPERIMENT_TAG}.npy"
val_embed_path = INTERIM_DATA_DIR / f"validation_doc_embedding_finbert_{DATASET_TAG}_{EXPERIMENT_TAG}.npy"
test_embed_path = INTERIM_DATA_DIR / f"test_doc_embedding_finbert_{DATASET_TAG}_{EXPERIMENT_TAG}.npy"

chunk_info_path = REPORTS_DIR / f"finbert_chunk_stats_{DATASET_TAG}_{EXPERIMENT_TAG}.json"

print(train_embed_path)
print(val_embed_path)
print(test_embed_path)

In [ ]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_doc_5d_large_lite.parquet")
val_df = pd.read_parquet(PROCESSED_DATA_DIR / "validation_doc_5d_large_lite.parquet")
test_df = pd.read_parquet(PROCESSED_DATA_DIR / "test_doc_5d_large_lite.parquet")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts(dropna=False).sort_index())

print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts(dropna=False).sort_index())

print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts(dropna=False).sort_index())

In [ ]:
DEBUG_SMALL = True

if DEBUG_SMALL:
    train_df = train_df.head(200)
    val_df = val_df.head(50)
    test_df = test_df.head(100)
    print("DEBUG_SMALL is ON")
    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)

In [ ]:
print(f"Loading tokenizer and model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.unk_token

model.to(DEVICE)
model.eval()

print("Tokenizer type:", type(tokenizer))
print("Model loaded.")

In [ ]:
def chunk_document(text: str, tokenizer):
    """
    Split a long document into chunks in a tokenizer-compatible way.

    Steps:
    1. tokenize the full text into wordpiece tokens
    2. split tokens into chunks of size CHUNK_BODY_LEN
    3. convert each chunk back to text
    4. run normal tokenizer() on each chunk
    """
    text = str(text)

    tokens = tokenizer.tokenize(text)

    chunks = []
    for i in range(0, len(tokens), CHUNK_BODY_LEN):
        if len(chunks) >= MAX_CHUNKS_PER_DOC:
            break

        chunk_tokens = tokens[i:i + CHUNK_BODY_LEN]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)

        encoded = tokenizer(
            chunk_text,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_attention_mask=True,
        )

        chunks.append({
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        })

    if len(chunks) == 0:
        encoded = tokenizer(
            "",
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_attention_mask=True,
        )
        chunks = [{
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        }]

    return chunks


def pad_batch(chunk_batch, pad_token_id):
    max_len = max(len(x["input_ids"]) for x in chunk_batch)

    input_ids = []
    attention_masks = []

    for x in chunk_batch:
        ids = x["input_ids"]
        mask = x["attention_mask"]

        pad_len = max_len - len(ids)
        ids = ids + [pad_token_id] * pad_len
        mask = mask + [0] * pad_len

        input_ids.append(ids)
        attention_masks.append(mask)

    return (
        torch.tensor(input_ids, dtype=torch.long, device=DEVICE),
        torch.tensor(attention_masks, dtype=torch.long, device=DEVICE),
    )


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

In [ ]:
def encode_documents(texts, split_name):
    all_doc_embeddings = []
    chunk_counts = []

    print(f"\nEncoding {split_name} documents on {DEVICE} ...")

    for text in tqdm(texts, desc=f"Encoding {split_name}"):
        chunks = chunk_document(text, tokenizer)
        chunk_counts.append(len(chunks))

        chunk_embeddings = []

        for i in range(0, len(chunks), BATCH_SIZE):
            chunk_batch = chunks[i:i + BATCH_SIZE]
            input_ids, attention_mask = pad_batch(chunk_batch, tokenizer.pad_token_id)

            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                pooled = mean_pool(outputs.last_hidden_state, attention_mask)

            chunk_embeddings.append(pooled.detach().cpu().numpy())

        chunk_embeddings = np.vstack(chunk_embeddings).astype(np.float32)
        doc_embedding = np.mean(chunk_embeddings, axis=0).astype(np.float32)
        all_doc_embeddings.append(doc_embedding)

    X = np.vstack(all_doc_embeddings).astype(np.float32)
    return X, chunk_counts

In [ ]:
if train_embed_path.exists():
    print("Loading cached train embeddings...")
    X_train = np.load(train_embed_path)
    train_chunk_counts = None
else:
    X_train, train_chunk_counts = encode_documents(
        train_df[TEXT_COL].astype(str).tolist(),
        "train"
    )
    np.save(train_embed_path, X_train)
    print("Saved train embeddings to:", train_embed_path)

print("Train embedding shape:", X_train.shape)

In [ ]:
if val_embed_path.exists():
    print("Loading cached validation embeddings...")
    X_val = np.load(val_embed_path)
    val_chunk_counts = None
else:
    X_val, val_chunk_counts = encode_documents(
        val_df[TEXT_COL].astype(str).tolist(),
        "validation"
    )
    np.save(val_embed_path, X_val)
    print("Saved validation embeddings to:", val_embed_path)

print("Validation embedding shape:", X_val.shape)

In [ ]:
if test_embed_path.exists():
    print("Loading cached test embeddings...")
    X_test = np.load(test_embed_path)
    test_chunk_counts = None
else:
    X_test, test_chunk_counts = encode_documents(
        test_df[TEXT_COL].astype(str).tolist(),
        "test"
    )
    np.save(test_embed_path, X_test)
    print("Saved test embeddings to:", test_embed_path)

print("Test embedding shape:", X_test.shape)

In [ ]:
y_train = train_df[LABEL_COL].astype(int).values
y_val = val_df[LABEL_COL].astype(int).values
y_test = test_df[LABEL_COL].astype(int).values

print(y_train.shape, y_val.shape, y_test.shape)

In [ ]:
def train_classifier(X_train, y_train, X_val, y_val):
    candidates = [
        {"C": 0.1},
        {"C": 1.0},
        {"C": 10.0},
    ]

    best_model = None
    best_result = None
    all_results = []

    for i, cfg in enumerate(candidates, start=1):
        print(f"\nTraining candidate {i}: {cfg}")

        clf = LogisticRegression(
            C=cfg["C"],
            max_iter=3000,
            random_state=42,
        )
        clf.fit(X_train, y_train)
        val_pred = clf.predict(X_val)

        acc = accuracy_score(y_val, val_pred)
        f1 = f1_score(y_val, val_pred)

        result = {
            "candidate_id": i,
            "C": cfg["C"],
            "val_accuracy": acc,
            "val_f1": f1,
        }
        all_results.append(result)

        print(f"Validation Accuracy: {acc:.4f}")
        print(f"Validation F1:       {f1:.4f}")

        if best_result is None or f1 > best_result["val_f1"]:
            best_result = result
            best_model = clf

    return best_model, best_result, pd.DataFrame(all_results)


best_model, best_result, validation_results_df = train_classifier(
    X_train, y_train, X_val, y_val
)

print("\nBest validation configuration:")
print(best_result)

In [ ]:
test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
cm = confusion_matrix(y_test, test_pred)
clf_report = classification_report(y_test, test_pred, output_dict=True)

print("\n=== TEST PERFORMANCE ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1:       {test_f1:.4f}")
print("Confusion Matrix:")
print(cm)

In [ ]:
joblib.dump(best_model, model_path)
validation_results_df.to_csv(val_table_path, index=False)

payload = {
    "dataset_tag": DATASET_TAG,
    "experiment_tag": EXPERIMENT_TAG,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "max_length": MAX_LENGTH,
    "max_chunks_per_doc": MAX_CHUNKS_PER_DOC,
    "best_validation_result": best_result,
    "test_results": {
        "test_accuracy": test_acc,
        "test_f1": test_f1,
        "confusion_matrix": cm.tolist(),
        "classification_report": clf_report,
    },
    "avg_train_chunks_per_doc": None if train_chunk_counts is None else float(np.mean(train_chunk_counts)),
    "avg_validation_chunks_per_doc": None if val_chunk_counts is None else float(np.mean(val_chunk_counts)),
    "avg_test_chunks_per_doc": None if test_chunk_counts is None else float(np.mean(test_chunk_counts)),
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("Saved outputs:")
print(model_path)
print(report_path)
print(val_table_path)
print(train_embed_path)
print(val_embed_path)
print(test_embed_path)

In [ ]:
payload